# 02 · Previsão mensal por variável
    Execute o notebook 01 primeiro. A análise usa previsões diretas por variável e horizonte
    com defasagens mensais. Compare o erro por horizonte e amplie a janela de validação
    antes de publicar resultados do estudo. O pickle é interno e deve ser carregado
    somente quando produzido pela própria pipeline, sob as mesmas versões das bibliotecas.

## Configurar caminhos

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/feynmans_legacy/analog_study.py").exists()
)
sys.path.insert(0, str(ROOT / 'src'))
DATA = ROOT / 'data/tratados/spec_analogos'
RUN = ROOT / 'data/tratados/analise_analogos'
RUN.mkdir(parents=True, exist_ok=True)

## Definir os alvos e os horizontes

In [ ]:
import json
import pickle

from feynmans_legacy.analog_study import publish_forecasts, train_forecasts

HORIZONTES = [1, 2, 3]
TARGETS = json.loads((RUN / "selecao.json").read_text(encoding="utf-8"))["alvos_previsao"]
history = pd.read_parquet(RUN / "terra.parquet")


## Treinar e comparar os métodos

In [ ]:
bundle, metrics = train_forecasts(history, TARGETS, HORIZONTES)


## Salvar o modelo e as métricas

In [ ]:
model_dir = RUN / "modelos"
model_dir.mkdir(exist_ok=True)
artifact = model_dir / "previsao_mensal.pkl"
with artifact.open("wb") as file:
    pickle.dump(bundle, file, protocol=pickle.HIGHEST_PROTOCOL)
metrics.to_parquet(RUN / "metricas_previsao.parquet", index=False)


## Conferir o erro por horizonte

In [ ]:
print(metrics.to_string(index=False))

Em execução agendada, use a função abaixo com o Parquet observado e o pickle.
    Ao atualizar o mês de corte, regenere o artefato e revise as métricas. O mapa pode
    ler o Parquet publicado sem carregar o pickle.

## Recarregar o modelo e publicar as previsões

In [ ]:
forecasts = publish_forecasts(history, artifact)
forecasts.to_parquet(RUN / "previsoes.parquet", index=False)


## Conferir os valores previstos

In [ ]:
print(forecasts.head().to_string(index=False))